# ДЗ 5. Тензоры: малоранговые представления (CP, Tucker, Tensor-Train)

Продолжение ДЗ 4, но для массивов с тремя и более измерениями. Матричное SVD не обобщается на
тензоры однозначным образом — существует несколько разных, не эквивалентных друг другу способов
обобщить понятие "ранга" и "разложения" на тензоры: **CP** (сумма ранг-1 тензоров, обобщение
SVD), **Tucker** (ядро + факторные матрицы по каждой моде, допускает разный ранг по разным
измерениям) и **Tensor-Train** (цепочка небольших 3-мерных "ядер", радикально не страдает от
проклятия размерности).

Используем библиотеку `tensorly` (`pip install tensorly`, либо уже установлена в Colab) с
PyTorch-бэкендом: `tensorly.set_backend('pytorch')`.

Один из сквозных примеров — **сжатие свёрточного слоя обученной сети через CP-разложение**
(задача 3.1) — это не учебная абстракция, а реальная техника ускорения CNN (Lebedev et al.,
2014), которая пригодится вам через два занятия, когда дойдём до тем про CNN.

In [ ]:
# =========================
# SETUP / ENVIRONMENT CHECK
# =========================
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("tensorly") is None:
    print("Устанавливаем tensorly ...")
    subprocess.check_call([sys.executable,"-m","pip","install","-q","tensorly"])

import torch
import numpy as np
import matplotlib.pyplot as plt
import tensorly as tl

tl.set_backend("pytorch")
torch.manual_seed(0)
np.random.seed(0)
print("✓ Python:",sys.version.split()[0])
print("✓ PyTorch:",torch.__version__)
print("✓ TensorLy:",tl.__version__)
print("\nСреда готова к выполнению ДЗ 5.")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import tensorly as tl
tl.set_backend('pytorch')
torch.manual_seed(0)

## Уровень 1. Базовый

### Задача 1.1. Развёртки тензора (unfolding) и многолинейный ранг

В отличие от матриц, у тензора нет единственного числа "ранг" — вместо этого говорят о
**многолинейном ранге**: наборе рангов $(r_1, \ldots, r_N)$ его развёрток (matricization) по
каждой моде. Развёртка по моде $n$ — это способ "распрямить" тензор в матрицу, взяв ось $n$ как
строки, а всё остальное — как столбцы.

1. Для случайного тензора формы $(4, 5, 6)$ реализуйте (или используйте `tensorly.unfold`)
   развёртку по каждой из 3 мод, убедитесь в правильности формы каждой развёртки.
2. Посчитайте ранг каждой развёртки через `torch.linalg.matrix_rank` — для случайного тензора
   все ранги должны быть максимальны (равны соответствующей размерности).
3. Постройте тензор с **заведомо заниженным** многолинейным рангом (через произведение малого
   "ядра" на факторные матрицы, `torch.einsum('ijk,ai,bj,ck->abc', ...)`) и убедитесь, что ранги
   развёрток действительно оказались меньше.

In [ ]:
from tensorly import unfold

# 1.1: случайный тензор и его multilinear rank.
X=torch.randn(4,5,6,dtype=torch.float64)
expected_shapes=[(4,30),(5,24),(6,20)]
ranks=[]
for mode,shape in enumerate(expected_shapes):
    M=unfold(X,mode)
    assert tuple(M.shape)==shape
    ranks.append(int(torch.linalg.matrix_rank(M)))
print("Формы unfolding:",expected_shapes)
print("Ранги:",ranks)
assert ranks==[4,5,6]

# Тензор с заранее заданным заниженным multilinear rank (2,3,2).
G=torch.randn(2,3,2,dtype=torch.float64)
A=torch.randn(4,2,dtype=torch.float64)
B=torch.randn(5,3,dtype=torch.float64)
C=torch.randn(6,2,dtype=torch.float64)
low=torch.einsum('ijk,ai,bj,ck->abc',G,A,B,C)
low_ranks=[int(torch.linalg.matrix_rank(unfold(low,mode))) for mode in range(3)]
print("Ранги low-rank тензора:",low_ranks)
assert low_ranks[0] <= 2 and low_ranks[1] <= 3 and low_ranks[2] <= 2
print("✓ Unfolding и multilinear rank проверены.")

**Письменный вывод.** Для случайного тензора $(4,5,6)$ ранги всех трёх unfoldings максимальны: $(4,5,6)$. Для специально построенного тензора через ядро $(2,3,2)$ соответствующие ранги не превышают эти значения, что демонстрирует управление многолинейным рангом через факторные матрицы.

### Задача 1.2. разложение Таккера

Tucker-разложение представляет тензор как **ядро** меньшего размера, "раздутое" факторными
матрицами по каждой моде: $T \approx \mathcal{G} \times_1 A \times_2 B \times_3 C$. В отличие от
CP, ранги по разным модам **независимы** — прямое обобщение многолинейного ранга из задачи 1.1.

1. Постройте тензор с известными Tucker-рангами (например, `(3, 4, 2)`, разными по каждой моде!).
2. Разложите его функцией `tensorly.decomposition.tucker` с разными наборами рангов: заниженным,
   точным и завышенным.
3. Сравните с CP-разложением того же тензора (задача 1.2) по числу параметров при сопоставимой
   ошибке восстановления — какой метод компактнее для этого конкретного тензора?

In [ ]:
from tensorly.decomposition import tucker, parafac
from tensorly.cp_tensor import cp_to_tensor
from tensorly.tucker_tensor import tucker_to_tensor

# Используем тот же тензор low из задачи 1.1.
tucker_rank=[2,3,2]
core,factors=tucker(low,rank=tucker_rank,init='svd',tol=1e-8,n_iter_max=100)
recon=tucker_to_tensor((core,factors))
err=float(torch.linalg.norm(low-recon)/torch.linalg.norm(low))
print(f"Tucker rank={tucker_rank}, relative error={err:.3e}")
assert err < 1e-8

# CP с несколькими rank: выбираем наименьший rank с достаточно малой ошибкой.
cp_results=[]
for r in [2,3,4,6,8,10,12]:
    cp=parafac(low,rank=r,init='svd',n_iter_max=300,tol=1e-7,random_state=0)
    cp_recon=cp_to_tensor(cp)
    cp_err=float(torch.linalg.norm(low-cp_recon)/torch.linalg.norm(low))
    cp_params=int(np.prod(cp[0].shape) if hasattr(cp[0],'shape') else 0)
    # More robust parameter count: weights R + sum I_n R.
    cp_params=r*(low.shape[0]+low.shape[1]+low.shape[2])+r
    cp_results.append((r,cp_err,cp_params))
    print(f"CP rank={r:2d}: error={cp_err:.3e}, params={cp_params}")

# Tucker parameter count: core + factors.
tucker_params=int(np.prod(tucker_rank)+4*tucker_rank[0]+5*tucker_rank[1]+6*tucker_rank[2])
print(f"Tucker params: {tucker_params}")
# We only claim a parameter comparison for a CP solution that actually reached the target tolerance.
valid=[x for x in cp_results if x[1] < 1e-5]
if valid:
    best=min(valid,key=lambda x:x[2])
    print(f"Smallest tested CP with error <1e-5: rank={best[0]}, params={best[2]}")
else:
    print("Ни один из протестированных CP-rank не достиг 1e-5; это тоже информативный результат для данного тензора.")
print("✓ Tucker reconstruction проверен; CP сравнение выполнено экспериментально.")

**Письменный вывод.** Tucker использует отдельный ранг для каждой моды, поэтому хорошо подходит для тензора с асимметричным multilinear rank. CP использует один общий rank и может потребовать другое число параметров для сопоставимой ошибки; в ноутбуке сравнение делается по фактически полученной ошибке и числу параметров, без предположения о заранее известном победителе.

## Уровень 2. Посложнее

### Задача 2.1. Сжатие "цветного изображения" через Tucker с разными рангами по модам

Цветное изображение — это естественный тензор 3-го порядка: `(высота, ширина, 3 канала)`.
Используйте Tucker с **сильно асимметричными** рангами: большими для пространственных измерений,
но не больше 3 для канала цвета (больше и не нужно — там ровно 3 канала).

1. Постройте (или сгенерируйте синтетически) изображение как тензор.
2. Сожмите через Tucker с рангами вида `(r, r, 3)` для нескольких `r`.
3. Постройте кривую "ошибка от степени сжатия" и визуализируйте несколько результатов.

In [ ]:
# Синтетическое RGB-изображение.
H,W=64,96
y,x=torch.meshgrid(torch.linspace(0,1,H,dtype=torch.float64),torch.linspace(0,1,W,dtype=torch.float64),indexing='ij')
img=torch.stack([
    0.5+0.5*torch.sin(2*np.pi*(2*x+y)),
    0.5+0.5*torch.cos(2*np.pi*(x+2*y)),
    torch.exp(-((x-0.65)**2+(y-0.45)**2)/0.04)
],dim=2).clamp(0,1)

from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
ranks=[4,8,16,32]
errors=[]
plt.figure(figsize=(5,4)); plt.imshow(img.numpy()); plt.title('Original'); plt.axis('off'); plt.show()
for r in ranks:
    core,factors=tucker(img,rank=[r,r,3],init='svd',tol=1e-6,n_iter_max=100)
    rec=tucker_to_tensor((core,factors)).clamp(0,1)
    err=float(torch.linalg.norm(img-rec)/torch.linalg.norm(img))
    errors.append(err)
    compression=img.numel()/(core.numel()+sum(f.numel() for f in factors))
    print(f"r={r:2d}: relative error={err:.4f}, compression={compression:.2f}x")
    if r in (4,16,32):
        plt.figure(figsize=(5,4)); plt.imshow(rec.numpy()); plt.title(f'Tucker ({r},{r},3)'); plt.axis('off'); plt.show()
assert errors[-1] <= errors[0] + 1e-8
plt.figure(figsize=(6,4)); plt.plot(ranks,errors,marker='o'); plt.xlabel('spatial rank r'); plt.ylabel('relative error'); plt.grid(True); plt.show()
print("✓ Tucker-сжатие с асимметричными рангами (r,r,3) проверено.")

**Письменный вывод.** В Tucker-компрессии пространственные ранги увеличиваются вместе с качеством восстановления, а цветовая мода оставляется равной 3. Поэтому при увеличении `r` ошибка обычно уменьшается, но число параметров и стоимость хранения растут.

### Задача 2.2. Tensor-Train и проклятие размерности

Полный тензор порядка $N$ с размером моды $d$ занимает $d^N$ элементов — при $N=20$, $d=8$ это
$\sim 10^{18}$ чисел, физически невозможно хранить. Tensor-Train представляет тензор как цепочку
малых 3-мерных "ядер" $G_1, \ldots, G_N$ (каждое — $(r_{n-1}, d, r_n)$), и число параметров растёт
**линейно** по $N$, а не экспоненциально.

1. Постройте тензор 6-го порядка (размер моды 8 по каждой из 6 осей) с известным TT-рангом,
   явно перемножив TT-ядра.
2. Разложите его функцией `tensorly.decomposition.tensor_train` с разными рангами, сравните
   ошибку восстановления и число параметров с полным размером тензора.
3. Постройте (аналитически, без реального построения тензора — это невозможно уже при небольшом
   числе мод!) таблицу/график: как растёт число элементов полного тензора и число параметров
   TT-разложения (при фиксированном ранге) с увеличением числа мод от 2 до 20.

In [ ]:
from tensorly.decomposition import tensor_train
from tensorly import tt_to_tensor

# TT-тензор 8^6 с известными TT-ranks 2.
d=8; N=6; tt_rank=2
G=torch.Generator().manual_seed(42)
cores=[torch.randn(1,d,tt_rank,generator=G,dtype=torch.float64)]
for _ in range(N-2):
    cores.append(torch.randn(tt_rank,d,tt_rank,generator=G,dtype=torch.float64))
cores.append(torch.randn(tt_rank,d,1,generator=G,dtype=torch.float64))

# Явно перемножаем TT-ядра; 8^6=262144, это ещё допустимо.
T=cores[0]
# Contract adjacent TT rank dimensions while preserving physical axes.
T=cores[0]
for core in cores[1:]:
    T=torch.tensordot(T,core,dims=([-1],[0]))
# Shape is (1,d,d,d,d,d,d,1); remove boundary rank dimensions.
T=T.squeeze(0).squeeze(-1)
assert tuple(T.shape)==(d,)*N

full_params=T.numel()
known_tt_params=sum(c.numel() for c in cores)
print(f"Полный тензор: {full_params:,} элементов")
print(f"Известные TT-ядра: {known_tt_params:,} параметров")

for rank in [1,2,3,4]:
    tt=tensor_train(T,rank=rank)
    rec=tt_to_tensor(tt)
    err=float(torch.linalg.norm(T-rec)/torch.linalg.norm(T))
    params=sum(c.numel() for c in tt.factors)
    print(f"TT rank={rank}: error={err:.3e}, params={params:,}, compression={full_params/params:.1f}x")
    if rank == 2:
        assert err < 1e-10

# Аналитический рост параметров при фиксированном TT-rank r=2.
ns=np.arange(2,21)
full_sizes=d**ns
# endpoints rank 1, internal rank r=2: 2dr + (N-2)dr^2
r=2
tt_sizes=2*d*r+(ns-2)*d*r*r
print("\nТаблица роста:")
for N_i,full_i,tt_i in zip(ns,full_sizes,tt_sizes):
    print(f"N={N_i:2d}: full={full_i:.3e}, TT(r=2)={tt_i:.0f}")

plt.figure(figsize=(7,4)); plt.semilogy(ns,full_sizes,label='full tensor'); plt.semilogy(ns,tt_sizes,label='TT, r=2'); plt.xlabel('number of modes N'); plt.ylabel('parameters'); plt.legend(); plt.grid(True); plt.show()
assert tt_sizes[-1] < full_sizes[-1]
print("✓ TT reconstruction и аналитический рост числа параметров проверены.")
print(f"Для N=20: полный тензор = {full_sizes[-1]:.3e} элементов, TT(r=2) = {tt_sizes[-1]:.0f} параметров.")

**Письменный вывод.** Полный тензор имеет $d^N$ элементов и растёт экспоненциально по числу мод. При фиксированном TT-rank число параметров TT растёт линейно по $N$: для $r=2$ оно равно $2dr+(N-2)dr^2$. На примере $8^6$ точное TT-разложение rank 2 восстанавливает исходный тензор с численной ошибкой около нуля.

---
**Что сдавать:** ноутбук с реализованными функциями, графиками и письменными ответами на вопросы
для решённых задач. Требуется `pip install tensorly` (в Colab одна команда, без дополнительной
настройки). Все вычисления в сумме — не больше нескольких минут на CPU.